# Tutorial: gather results by unit combo

This notebook shows the process for taking the results by LSOA and gathering them by combination of stroke units across the scenarios.

## Code setup

In [38]:
import os
import numpy as np
import pandas as pd

# Add path to "classes" directory:
import sys
sys.path.append('../')

from classes.calc_by_unit import calculate_results_per_scenario, \
    calculate_extra_bits

In [2]:
dir_data = os.path.join('..', 'data')
dir_output = os.path.join('..', 'output')
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
file_unit_combos = 'unit_combos.csv'
# Output:
file_comparison_unit_combo = 'comparison_by_unit_combo_base_taunworc'

In [3]:
strings_tl = ['', '_transferloss0!67']  # column names for transfer rates
scenarios = ['base', 'taunworc']

In [4]:
scenario = 'base'
s_bias = ''

## Load data

### Hospital reference

In [5]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [6]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [7]:
print(len(df_hospitals))

131


In [8]:
print(len(df_hospitals.index) == len(df_hospitals.index.unique()))

True


### Unit combinations


In [9]:
p = os.path.join(dir_output, file_unit_combos)
df_unit_combos = pd.read_csv(p)

In [10]:
df_unit_combos.head(3)

,unit_combo,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min,bias15min
0,42000,BN25BE,BN25BE,BN25BE,BN25BE,1,1,1
1,42001,BN112DH,BN112DH,BN25BE,BN25BE,1,1,1
2,42002,CA27HY,CA27HY,NE14LP,NE14LP,1,1,1


Gather SSNAP names and hospital cities too:

In [11]:
cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit']
cols_units_here = [f'{c}_{s}' for c in cols_unit_postcodes
                   for s in scenarios]

In [12]:
cols_hosp = ['SSNAP name', 'Hospital_City']
for unit_col in cols_units_here:
    # Set up names for new columns:
    t = '_transfer_unit' if 'transfer' in unit_col else ''
    s = unit_col.split('_')[-1]  # either 'base' or 'taunworc'
    cols_new = [f'{c}{t}_{s}' for c in cols_hosp]
    rename_dict = dict(zip(cols_hosp, cols_new))
    # Merge in unit names:
    df_unit_combos = pd.merge(
        df_unit_combos,
        df_hospitals[cols_hosp].rename(columns=rename_dict).reset_index(),
        left_on=unit_col,
        right_on='Postcode',
        how='left'
    ).drop('Postcode', axis='columns')

Check results:

In [13]:
df_unit_combos.head(3)

,unit_combo,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min,bias15min,SSNAP name_base,Hospital_City_base,SSNAP name_taunworc,Hospital_City_taunworc,SSNAP name_transfer_unit_base,Hospital_City_transfer_unit_base,SSNAP name_transfer_unit_taunworc,Hospital_City_transfer_unit_taunworc
0,42000,BN25BE,BN25BE,BN25BE,BN25BE,1,1,1,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
1,42001,BN112DH,BN112DH,BN25BE,BN25BE,1,1,1,Worthing Hospital,Worthing,Worthing Hospital,Worthing,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
2,42002,CA27HY,CA27HY,NE14LP,NE14LP,1,1,1,Cumberland Infirmary,Carlisle,Cumberland Infirmary,Carlisle,Royal Victoria Infirmary,Newcastle,Royal Victoria Infirmary,Newcastle


In [14]:
df_unit_combos.columns

Index(['unit_combo', 'nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc',
       'transfer_unit_base', 'transfer_unit_taunworc', 'standardbias',
       'bias10min', 'bias15min', 'SSNAP name_base', 'Hospital_City_base',
       'SSNAP name_taunworc', 'Hospital_City_taunworc',
       'SSNAP name_transfer_unit_base', 'Hospital_City_transfer_unit_base',
       'SSNAP name_transfer_unit_taunworc',
       'Hospital_City_transfer_unit_taunworc'],
      dtype='str')

### Base scenario results

In [15]:
# Load in the LSOA-level outcomes and admissions data:
p = os.path.join(dir_output, file_results_comparison + s_bias + '.csv')
df = pd.read_csv(p, index_col=0)

# Merge unit combination labels into the main dataframe:
df = pd.merge(
    df.reset_index(), df_unit_combos, on=cols_units_here, how='left'
).set_index('LSOA')

In [16]:
display(df.head(3))

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,transfer_required_base,drip_ship_mt_time_base,drip_ship_mt_travel_time_base,drip_ship_mt_travel_time_with_delay_base,admissions_mt_base,transfers_mt_base,...,bias10min,bias15min,SSNAP name_base,Hospital_City_base,SSNAP name_taunworc,Hospital_City_taunworc,SSNAP name_transfer_unit_base,Hospital_City_transfer_unit_base,SSNAP name_transfer_unit_taunworc,Hospital_City_transfer_unit_taunworc
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,Adur 001A,1.49403,BN25BE,BN25BE,False,181,11,11,0.149,0.000,...,1,1,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
ADUR001B,Adur 001B,1.86762,BN25BE,BN25BE,False,182,12,12,0.187,0.000,...,1,1,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
ADUR001C,Adur 001C,2.24110,BN112DH,BN25BE,True,265,35,95,0.224,0.224,...,1,1,Worthing Hospital,Worthing,Worthing Hospital,Worthing,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton


In [17]:
print(df.columns)

Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'transfer_unit_base', 'transfer_required_base',
       'drip_ship_mt_time_base', 'drip_ship_mt_travel_time_base',
       'drip_ship_mt_travel_time_with_delay_base', 'admissions_mt_base',
       'transfers_mt_base', 'admissions_mt_transferloss0!67_base',
       'transfers_mt_transferloss0!67_base',
       'admissions_weighted_drip_ship_mrs_0-2_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base',
       'drip_ship_mrs_0-2_base', 'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_mrs_0-2_transferloss0!67_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base',
       'nearest_ivt_unit_taunworc', 'transfer_unit_taunworc',
       'transfer_required_taunworc', 'drip_ship_mt_time_taunworc',
       'drip_ship_mt_travel_time_taunworc',
   

## Set up column names

Gather columns present in the main results dataframes.

Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [18]:
col_lsoa11nm = 'area_lower'
col_admissions = 'Admissions_2025_Est'
cols_mt_admissions = ['admissions_mt']
cols_transfer = ['transfers_mt']
cols_time = ['drip_ship_mt_time',
             'drip_ship_mt_travel_time',
             'drip_ship_mt_travel_time_with_delay']
cols_outcome = [
    'drip_ship_mrs_0-2',
    'lvo_drip_ship_ivt_mt_mrs_0-2',
]
cols_admissions_weighted_outcomes = [
    'admissions_weighted_drip_ship_mrs_0-2',
    'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
]
cols_change = ['change_in_drip_ship_mrs_0-2',
               'change_in_lvo_drip_ship_ivt_mt_mrs_0-2']


# Add columns affected by lossy transfer rate:
def add_str_to_columns(cols, s_list):
    return [f'{c}{s}' for c in cols for s in s_list]
    
cols_mt_admissions = add_str_to_columns(cols_mt_admissions, strings_tl)
cols_transfer = add_str_to_columns(cols_transfer, strings_tl)
cols_outcome = add_str_to_columns(cols_outcome, strings_tl)
cols_admissions_weighted_outcomes = add_str_to_columns(
    cols_admissions_weighted_outcomes, strings_tl)
cols_change = add_str_to_columns(cols_change, strings_tl)

# Final gather:
cols_change += [
    'change_in_drip_ship_mt_time',
    'change_in_drip_ship_mt_travel_time',
    'change_in_drip_ship_mt_travel_time_with_delay',
    'changed'
]
cols_shared = [col_lsoa11nm, col_admissions] + cols_change

## Pick out base columns

Pick out just the columns for the base scenario, not `_taunworc`:

In [19]:
cols = (['unit_combo'] + cols_shared +
        [c for c in df.columns if c.endswith(scenario)])
df_here = df[cols]
rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                    for c in df_here.columns])
df_here = df_here.rename(columns=rename_dict)

In [20]:
print(df_here.columns)

Index(['unit_combo', 'area_lower', 'Admissions_2025_Est',
       'change_in_drip_ship_mrs_0-2',
       'change_in_drip_ship_mrs_0-2_transferloss0!67',
       'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
       'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67',
       'change_in_drip_ship_mt_time', 'change_in_drip_ship_mt_travel_time',
       'change_in_drip_ship_mt_travel_time_with_delay', 'changed',
       'nearest_ivt_unit', 'transfer_unit', 'transfer_required',
       'drip_ship_mt_time', 'drip_ship_mt_travel_time',
       'drip_ship_mt_travel_time_with_delay', 'admissions_mt', 'transfers_mt',
       'admissions_mt_transferloss0!67', 'transfers_mt_transferloss0!67',
       'admissions_weighted_drip_ship_mrs_0-2',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67',
       'drip_ship_mrs_0-2', 'lvo_drip_ship_ivt_mt_mrs_0-2',
       'd

## Unit names

In [21]:
# Link each first unit to its MT unit.
# Only keep data for this scenario:
cols_here = ['unit_combo'] + [c for c in df_unit_combos if scenario in c]
df_unit_info = df_unit_combos[cols_here].set_index('unit_combo').copy()
# Only keep combos in the data here:
combos_here = df['unit_combo'].unique()
df_unit_info = df_unit_info[df_unit_info.index.isin(combos_here)]
# Rename columns to match data here:
rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                    for c in df_unit_info.columns])
df_unit_info = df_unit_info.rename(columns=rename_dict)

In [22]:
print('df_unit_info:', df_unit_info.columns)

df_unit_info: Index(['nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')


In [23]:
display(df_unit_info.head(3))

,nearest_ivt_unit,transfer_unit,SSNAP name,Hospital_City,SSNAP name_transfer_unit,Hospital_City_transfer_unit
unit_combo,,,,,,
42000,BN25BE,BN25BE,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
42001,BN112DH,BN25BE,Worthing Hospital,Worthing,Royal Sussex County Hospital,Brighton
42002,CA27HY,NE14LP,Cumberland Infirmary,Carlisle,Royal Victoria Infirmary,Newcastle


## Admissions

Admissions from multiple LSOA to the same unit can just be summed.

In [24]:
cols_admissions = [col_admissions] + cols_mt_admissions + cols_transfer
df_admissions = (
    df_here[['unit_combo'] + cols_admissions].groupby('unit_combo').sum())

df_admissions = df_admissions.fillna(0.0)
df_admissions = df_admissions.round(1)

In [25]:
print('df_admissions:', df_admissions.columns)

df_admissions: Index(['Admissions_2025_Est', 'admissions_mt',
       'admissions_mt_transferloss0!67', 'transfers_mt',
       'transfers_mt_transferloss0!67'],
      dtype='str')


In [26]:
display(df_admissions.head(3))

,Admissions_2025_Est,admissions_mt,admissions_mt_transferloss0!67,transfers_mt,transfers_mt_transferloss0!67
unit_combo,,,,,
42000,654.1,65.4,65.4,0.0,0.0
42001,584.5,58.4,39.1,58.4,39.1
42002,679.0,67.9,45.5,67.9,45.5


## Admissions-weighted outcomes

Sum the outcomes of all patients in the same unit to find the total number of patients with mRS 0-2. Then divide these by the direct admissions to each unit to find the proportion of patients with mRS 0-2.

In [27]:
# First sum by all patients:
df_unit_outcomes = (
    df_here[['unit_combo', col_admissions] +
            cols_admissions_weighted_outcomes]
    .groupby('unit_combo').sum()
)
# Divide outcomes by the direct admissions to each unit:
df_unit_outcomes[cols_admissions_weighted_outcomes] = (
    df_unit_outcomes[cols_admissions_weighted_outcomes]
        .div(df_unit_outcomes[col_admissions], axis='rows')
)
df_unit_outcomes = df_unit_outcomes.drop(col_admissions, axis='columns')
df_unit_outcomes = df_unit_outcomes.round(5)

In [28]:
print('df_unit_outcomes:', df_unit_outcomes.columns)

df_unit_outcomes: Index(['admissions_weighted_drip_ship_mrs_0-2',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'],
      dtype='str')


In [29]:
display(df_unit_outcomes.head(3))

,admissions_weighted_drip_ship_mrs_0-2,admissions_weighted_drip_ship_mrs_0-2_transferloss0!67,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67
unit_combo,,,,
42000,0.61838,0.61838,0.42841,0.42841
42001,0.60376,0.59866,0.37688,0.35985
42002,0.58891,0.58673,0.34305,0.33572


## Admissions-weighted travel times

Similarly to good outcomes: first calculate the total time across all patients going to a unit, and then divide by the total admissions to that unit to find the average time.

In [30]:
# First sum by all patients:
df_unit_times = df_here.copy()
df_unit_times[cols_time] = df_unit_times[cols_time].multiply(
                           df_unit_times[col_admissions], axis='rows')

df_unit_times = (
    df_unit_times[['unit_combo', col_admissions] + cols_time]
    .groupby('unit_combo').sum()
)
df_unit_times[cols_time] = df_unit_times[cols_time].div(
                           df_unit_times[col_admissions], axis='rows')

df_unit_times = df_unit_times.drop(col_admissions, axis='columns')
df_unit_times = df_unit_times.round(1)

In [31]:
print('df_unit_times:', df_unit_times.columns)

df_unit_times: Index(['drip_ship_mt_time', 'drip_ship_mt_travel_time',
       'drip_ship_mt_travel_time_with_delay'],
      dtype='str')


In [32]:
display(df_unit_times.head(3))

,drip_ship_mt_time,drip_ship_mt_travel_time,drip_ship_mt_travel_time_with_delay
unit_combo,,,
42000,182.5,12.5,12.5
42001,263.9,33.9,93.9
42002,319.8,89.8,149.8


## Gather all results

In [42]:
comparison_by_unit_combo_base = pd.concat(
    (df_unit_info, df_admissions, df_unit_outcomes, df_unit_times),
    axis='columns'
)

In [43]:
print(comparison_by_unit_combo_base.columns)

Index(['nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit',
       'Admissions_2025_Est', 'admissions_mt',
       'admissions_mt_transferloss0!67', 'transfers_mt',
       'transfers_mt_transferloss0!67',
       'admissions_weighted_drip_ship_mrs_0-2',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67',
       'drip_ship_mt_time', 'drip_ship_mt_travel_time',
       'drip_ship_mt_travel_time_with_delay'],
      dtype='str')


In [44]:
display(comparison_by_unit_combo_base.head(3))

,nearest_ivt_unit,transfer_unit,SSNAP name,Hospital_City,SSNAP name_transfer_unit,Hospital_City_transfer_unit,Admissions_2025_Est,admissions_mt,admissions_mt_transferloss0!67,transfers_mt,transfers_mt_transferloss0!67,admissions_weighted_drip_ship_mrs_0-2,admissions_weighted_drip_ship_mrs_0-2_transferloss0!67,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2,admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67,drip_ship_mt_time,drip_ship_mt_travel_time,drip_ship_mt_travel_time_with_delay
unit_combo,,,,,,,,,,,,,,,,,,
42000,BN25BE,BN25BE,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,654.1,65.4,65.4,0.0,0.0,0.61838,0.61838,0.42841,0.42841,182.5,12.5,12.5
42001,BN112DH,BN25BE,Worthing Hospital,Worthing,Royal Sussex County Hospital,Brighton,584.5,58.4,39.1,58.4,39.1,0.60376,0.59866,0.37688,0.35985,263.9,33.9,93.9
42002,CA27HY,NE14LP,Cumberland Infirmary,Carlisle,Royal Victoria Infirmary,Newcastle,679.0,67.9,45.5,67.9,45.5,0.58891,0.58673,0.34305,0.33572,319.8,89.8,149.8


## Conclusion

We have shown how the LSOA-level results are gathered into the results by combination of stroke units.